# 05 — Simulation ingestion live NDJSON → Silver Iceberg

**Objectif** : simuler l'arrivée des fichiers NDJSON IoT Hub
et écrire chaque batch dans Silver Iceberg  
**Pattern** : 1 fichier NDJSON = 1 batch 60s = 1 append Iceberg = 1 snapshot  
**Input** : `data/ndjson_sim/*.json`  
**Output** : table Iceberg `silver.trays` mise à jour

In [1]:
import os
import json
import base64
import hashlib
import re
import time
import warnings
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import pyarrow as pa
from pyiceberg.catalog.sql import SqlCatalog
from pyiceberg.expressions import And, EqualTo
from dotenv import load_dotenv

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)

load_dotenv()
print("OK")

OK


In [2]:
import sys
sys.path.append("../src")
from catalog import get_catalog, ADLS_URI
from schema import get_or_create_silver_trays

catalog = get_catalog()
table   = get_or_create_silver_trays(catalog, ADLS_URI)

Table 'silver.trays' créée.
  Location : abfss://silver@dlsecatcandlingfrcedev.dfs.core.windows.net/trays_iceberg
  Format   : Iceberg v2
  Champs   : 27


In [3]:
RE_MATRIX = re.compile(r"\[(\d+)\]\[(\d+)\]$")

def decode_message(raw_line: str) -> dict:
    envelope = json.loads(raw_line)
    body     = json.loads(base64.b64decode(envelope["Body"]).decode("utf-8"))
    return {
        "enqueued_at": envelope["EnqueuedTimeUtc"],
        "device_id":   envelope["SystemProperties"]["connectionDeviceId"],
        "machine_id":  body["process_serial_number"],
        "box":         body["box"],
        "values": {
            v["id"].split(".")[-1]: v["value"]
            for v in body["values"]
        },
        "timestamps": {
            v["id"].split(".")[-1]: v["timestamp"]
            for v in body["values"]
        }
    }

def extract_matrix(values: dict, tag_prefix: str) -> list:
    matrix = [[0] * 10 for _ in range(15)]
    for tag_name, val in values.items():
        if not tag_name.startswith(tag_prefix):
            continue
        m = RE_MATRIX.search(tag_name)
        if not m:
            continue
        row = int(m.group(1)) - 1
        col = int(m.group(2)) - 1
        if 0 <= row < 15 and 0 <= col < 10:
            matrix[row][col] = int(val) if val is not None else 0
    return matrix

def matrix_to_compact(matrix: list) -> str:
    return "".join(str(cell) for row in matrix for cell in row)

def compute_counts(matrix: list) -> dict:
    flat = [cell for row in matrix for cell in row]
    return {
        "n_total":      len(flat),
        "n_fertile":    flat.count(1),
        "n_clear":      flat.count(3),
        "n_early_dead": flat.count(2),
        "n_late_dead":  flat.count(4),
        "n_missing":    flat.count(0),
    }

def compute_tray_id(machine_id: str, candled_at_utc: str) -> str:
    raw = f"{machine_id}|{candled_at_utc}"
    return hashlib.sha256(raw.encode("utf-8")).hexdigest()

def parse_ndjson_file(filepath: Path) -> pd.DataFrame:
    rows = []
    with open(filepath, "r", encoding="utf-8") as f:
        lines = [l.strip() for l in f if l.strip()]

    for line in lines:
        msg = decode_message(line)

        if msg["values"].get("pmaf_trig") != "Tray":
            continue

        v              = msg["values"]
        candled_at_str = msg["timestamps"].get("pmaf_trig", msg["enqueued_at"])
        candled_at     = pd.to_datetime(candled_at_str, utc=True)
        matrix         = extract_matrix(v, "final_candled_eggs")
        counts         = compute_counts(matrix)
        light_mat      = extract_matrix(v, "laser1_light_transmitted_eggs")
        light_flat     = [cell for row in light_mat for cell in row]

        rows.append({
            "tray_id":                  compute_tray_id(msg["machine_id"], candled_at_str),
            "machine_id":               msg["machine_id"],
            "candled_at":               candled_at,
            "candled_date":             candled_at.date(),
            "flock":                    int(v.get("flock_number", 0)),
            "trolley":                  str(v.get("trolley_name", "")),
            "tray_seq":                 int(v.get("setter_tray_number_flock", 0)),
            "flock_name":               str(v.get("flock_name", "")),
            "trolley_name":             str(v.get("trolley_name", "")),
            "caliber":                  str(v.get("caliber", "")),
            "setter_tray_number_flock": int(v.get("setter_tray_number_flock", 0)),
            **counts,
            "is_count_consistent":      counts["n_total"] == 150,
            "matrix_compact":           matrix_to_compact(matrix),
            "light_flat":               light_flat,
            "alarm_emergency_stop":     int(v.get("alarm_emergency_stop", 0)),
            "alarm_air_pressure_fault": int(v.get("alarm_air_pressure_fault", 0)),
            "processed_at":             pd.Timestamp.utcnow(),
            "bronze_path":              filepath.name,
            "year":                     candled_at.year,
            "month":                    candled_at.month,
            "day":                      candled_at.day,
        })

    return pd.DataFrame(rows)

print("Fonctions de parsing définies.")

Fonctions de parsing définies.


In [4]:
def df_to_arrow(df: pd.DataFrame) -> pa.Table:
    def clean_light_flat(val):
        if val is None:
            return None
        return [0 if (v is None or (isinstance(v, float) and np.isnan(v)))
                else int(v) for v in val]

    df = df.copy()
    df["light_flat"]               = df["light_flat"].apply(clean_light_flat)
    df["candled_at"]               = pd.to_datetime(df["candled_at"], utc=True)
    df["processed_at"]             = pd.to_datetime(df["processed_at"], utc=True)
    df["candled_date"]             = pd.to_datetime(df["candled_date"]).dt.date
    df["flock"]                    = df["flock"].astype("int32")
    df["trolley"]                  = df["trolley"].astype("str")
    df["tray_seq"]                 = df["tray_seq"].astype("int32")
    df["setter_tray_number_flock"] = df["setter_tray_number_flock"].fillna(0).astype("int32")
    df["n_total"]                  = df["n_total"].astype("int32")
    df["n_fertile"]                = df["n_fertile"].astype("int32")
    df["n_clear"]                  = df["n_clear"].astype("int32")
    df["n_early_dead"]             = df["n_early_dead"].astype("int32")
    df["n_late_dead"]              = df["n_late_dead"].astype("int32")
    df["n_missing"]                = df["n_missing"].astype("int32")
    df["alarm_emergency_stop"]     = df["alarm_emergency_stop"].fillna(0).astype("int32")
    df["alarm_air_pressure_fault"] = df["alarm_air_pressure_fault"].fillna(0).astype("int32")
    df["is_count_consistent"]      = df["is_count_consistent"].astype("bool")
    df["year"]                     = df["year"].astype("int32")
    df["month"]                    = df["month"].astype("int32")
    df["day"]                      = df["day"].astype("int32")

    arrow_schema = pa.schema([
        pa.field("tray_id",                  pa.string(),               nullable=False),
        pa.field("machine_id",               pa.string(),               nullable=False),
        pa.field("candled_at",               pa.timestamp("us", "UTC"), nullable=False),
        pa.field("candled_date",             pa.date32(),               nullable=False),
        pa.field("flock",                    pa.int32(),                nullable=False),
        pa.field("trolley",                  pa.string(),               nullable=False),
        pa.field("tray_seq",                 pa.int32(),                nullable=False),
        pa.field("flock_name",               pa.string(),               nullable=True),
        pa.field("trolley_name",             pa.string(),               nullable=True),
        pa.field("caliber",                  pa.string(),               nullable=True),
        pa.field("setter_tray_number_flock", pa.int32(),                nullable=True),
        pa.field("n_total",                  pa.int32(),                nullable=False),
        pa.field("n_fertile",                pa.int32(),                nullable=False),
        pa.field("n_clear",                  pa.int32(),                nullable=False),
        pa.field("n_early_dead",             pa.int32(),                nullable=False),
        pa.field("n_late_dead",              pa.int32(),                nullable=False),
        pa.field("n_missing",                pa.int32(),                nullable=False),
        pa.field("is_count_consistent",      pa.bool_(),                nullable=False),
        pa.field("matrix_compact",           pa.string(),               nullable=False),
        pa.field("light_flat",               pa.list_(pa.int32()),      nullable=True),
        pa.field("alarm_emergency_stop",     pa.int32(),                nullable=True),
        pa.field("alarm_air_pressure_fault", pa.int32(),                nullable=True),
        pa.field("processed_at",             pa.timestamp("us", "UTC"), nullable=False),
        pa.field("bronze_path",              pa.string(),               nullable=True),
        pa.field("year",                     pa.int32(),                nullable=False),
        pa.field("month",                    pa.int32(),                nullable=False),
        pa.field("day",                      pa.int32(),                nullable=False),
    ])

    return pa.Table.from_pandas(df, schema=arrow_schema, preserve_index=False)

print("Fonction de conversion définie.")

Fonction de conversion définie.


In [5]:
NDJSON_DIR   = Path("../data/ndjson_sim")
SIMULATE_LAG = False  # True = attendre 5s entre chaque fichier, False = tout d'un coup

files = sorted(NDJSON_DIR.glob("**/*.json"))
print(f"Fichiers à ingérer : {len(files)}")
print(f"Mode simulation lag : {SIMULATE_LAG}\n")

results = []

for i, filepath in enumerate(files):
    t_start = time.time()

    # 1. Parser le fichier NDJSON
    df_batch = parse_ndjson_file(filepath)

    if df_batch.empty:
        print(f"[{i+1:03d}/{len(files)}] {filepath.name} — aucun plateau, skip")
        continue

    # 2. Convertir en PyArrow
    arrow_batch = df_to_arrow(df_batch)

    # 3. Append dans Iceberg
    table.append(arrow_batch)

    t_elapsed = time.time() - t_start
    snapshot  = table.current_snapshot()

    results.append({
        "file":        filepath.name,
        "plateaux":    len(df_batch),
        "snapshot_id": snapshot.snapshot_id,
        "elapsed_s":   round(t_elapsed, 2),
    })

    print(
        f"[{i+1:03d}/{len(files)}] {filepath.name} — "
        f"{len(df_batch)} plateaux — "
        f"snapshot {snapshot.snapshot_id} — "
        f"{t_elapsed:.2f}s"
    )

    if SIMULATE_LAG:
        time.sleep(5)

print(f"\nIngestion terminée — {len(results)} fichiers traités.")

Fichiers à ingérer : 180
Mode simulation lag : False

[001/180] 04_32_0.json — 2 plateaux — snapshot 3686287724106525192 — 0.67s
[002/180] 04_33_0.json — 11 plateaux — snapshot 8813609995095537524 — 0.62s
[003/180] 04_34_0.json — 9 plateaux — snapshot 5270281244342531051 — 0.48s
[004/180] 04_35_0.json — 6 plateaux — snapshot 2633451521130558220 — 0.47s
[005/180] 04_36_0.json — 4 plateaux — snapshot 8173246843929870836 — 0.47s
[006/180] 04_37_0.json — 9 plateaux — snapshot 891038821813661921 — 0.48s
[007/180] 04_38_0.json — 10 plateaux — snapshot 8878300186691725336 — 0.53s
[008/180] 04_39_0.json — 10 plateaux — snapshot 8385436421705579161 — 0.58s
[009/180] 04_40_0.json — 9 plateaux — snapshot 58684751662899023 — 0.49s
[010/180] 04_41_0.json — 9 plateaux — snapshot 329825600385903564 — 0.48s
[011/180] 04_42_0.json — 9 plateaux — snapshot 3075382244944036812 — 0.76s
[012/180] 04_43_0.json — 8 plateaux — snapshot 425841991316064088 — 0.44s
[013/180] 04_44_0.json — 10 plateaux — snapshot 

In [6]:
df_results = pd.DataFrame(results)

print("=== Résumé ingestion ===")
print(f"Fichiers traités  : {len(df_results)}")
print(f"Total plateaux    : {df_results['plateaux'].sum()}")
print(f"Snapshots créés   : {len(table.snapshots())}")
print(f"Latence moy/batch : {df_results['elapsed_s'].mean():.2f}s")
print(f"Latence max/batch : {df_results['elapsed_s'].max():.2f}s")
print()
df_results

=== Résumé ingestion ===
Fichiers traités  : 180
Total plateaux    : 1344
Snapshots créés   : 180
Latence moy/batch : 0.59s
Latence max/batch : 1.46s



,file,plateaux,snapshot_id,elapsed_s
0,04_32_0.json,2,3686287724106525192,0.67
1,04_33_0.json,11,8813609995095537524,0.62
2,04_34_0.json,9,5270281244342531051,0.48
3,04_35_0.json,6,2633451521130558220,0.47
4,04_36_0.json,4,8173246843929870836,0.47
...,...,...,...,...
175,07_53_0.json,7,4869989814171654114,0.61
176,07_54_0.json,8,2281699091183902801,0.67
177,07_55_0.json,1,4752602484732368314,0.68
178,07_56_0.json,7,747198362545679719,0.63


In [7]:
from pyiceberg.expressions import And, EqualTo

MACHINE_ID = "PMAF-C012501"

df_final = table.scan(
    row_filter=And(
        EqualTo("machine_id", MACHINE_ID),
        EqualTo("year",  2026),
        EqualTo("month", 5),
        EqualTo("day",   15),
    )
).to_pandas()

print(f"Total lignes relues depuis Iceberg : {len(df_final)}")
print(f"Flocks distincts : {sorted(df_final['flock'].unique())}")
print(f"Trolleys distincts : {df_final['trolley'].nunique()}")
print(f"is_count_consistent=True  : {df_final['is_count_consistent'].sum()}")
print(f"is_count_consistent=False : {(~df_final['is_count_consistent']).sum()}")
print()
df_final.groupby("flock")[["n_total", "n_fertile", "n_clear"]].sum().sort_index().head(10)

Total lignes relues depuis Iceberg : 1344
Flocks distincts : [np.int32(1), np.int32(2), np.int32(3), np.int32(4), np.int32(5), np.int32(6), np.int32(7), np.int32(8), np.int32(9), np.int32(10), np.int32(11), np.int32(12), np.int32(13), np.int32(14), np.int32(15), np.int32(16), np.int32(17), np.int32(18), np.int32(19), np.int32(20), np.int32(21), np.int32(22), np.int32(23), np.int32(24), np.int32(25), np.int32(26), np.int32(27), np.int32(28), np.int32(29), np.int32(30), np.int32(31), np.int32(32), np.int32(33), np.int32(34), np.int32(35), np.int32(36), np.int32(37), np.int32(38), np.int32(39), np.int32(40), np.int32(41), np.int32(42)]
Trolleys distincts : 41
is_count_consistent=True  : 1344
is_count_consistent=False : 0



,n_total,n_fertile,n_clear
flock,,,
1,4800,4402,311
2,4800,4403,307
3,4800,4375,334
4,4800,4392,299
5,4350,3996,284
6,6450,5903,435
7,4800,3974,679
8,4950,4043,775
9,4800,3923,708
